1. Setup and Imports

In [ ]:
import ast
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Import your custom components
from Representation import tfidf_representation
from LogisticRegressionModel import LogisticRegressionModel
from SVMModel import SVMModel
from NaiveBayesModel import NaiveBayesModel

In [ ]:
# Configuration
DATA_ROOT = "../../data"  # Update this path
DEBUG = 1

2. Data Loading and Preparation

In [ ]:
def load_data(data_path):
    """Load and preprocess the dataset"""
    df = pd.read_csv(data_path)
    df = df[df['review'].notna()]

    if DEBUG:
        print(f"Dataset loaded with {len(df)} samples")
        print("Sample before processing:")
        print(df['review'].iloc[0][:100] + "...")

    # Convert string representation of tokens to actual list
    df['review'] = df['review'].apply(ast.literal_eval)

    # Join tokens to form space-separated strings
    df['review'] = df['review'].apply(lambda tokens: " ".join(tokens))

    if DEBUG:
        print("\nSample after processing:")
        print(df['review'].iloc[0][:100] + "...")

    return df['review'].tolist(), df['sentiment'].tolist()

In [ ]:
# Load the data
texts, labels = load_data(f"{DATA_ROOT}/project2_data/cleaned_imdb_reviews.csv")

3. Text Representation

In [ ]:
X, vectorizer = tfidf_representation(texts)
y = labels

if DEBUG:
    print(f"\nCreated representation with {X.shape[0]} samples and {X.shape[1]} features")

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

 4. Model Training

In [ ]:

def create_model(model_type):
    """Factory function to create the appropriate model"""
    if model_type == "logistic":
        return LogisticRegressionModel(
            C=4.0,
            max_iter=10000,
            solver='liblinear',
            random_state=42,
            verbose=DEBUG
        )
    elif model_type == "svm":
        return SVMModel(
            kernel='linear',
            C=1.0,
            random_state=42,
            verbose=DEBUG
        )
    elif model_type == "naive_bayes":
        return NaiveBayesModel(
            variant='gaussian',
            alpha=1.0,
            random_state=42,
            verbose=DEBUG
        )
    else:
        raise ValueError(f"Unknown model type: {model_type}")

In [ ]:
MODEL_TYPE = "svm"  # Options: "logistic", "svm", "naive_bayes"

# Create and train model
model = create_model(MODEL_TYPE)
print(f"\nTraining {MODEL_TYPE} model...")

In [ ]:
X_train_cropped = X_train[:1000]  # For debugging, use a smaller subset
y_train_cropped = y_train[:1000]  # For debugging, use a smaller subset

In [ ]:
model.train(X_train_cropped, y_train_cropped)
# model.train(X_train, y_train)  # Uncomment for full training

 5. Evaluation

In [ ]:
def classification_report_percent(y_true, y_pred, target_names=None):
    report_dict = classification_report(y_true, y_pred, target_names=target_names, output_dict=True)
    lines = []
    headers = ["precision", "recall", "f1-score", "support"]
    line_fmt = "{:>9} {:>9} {:>9} {:>9}"

    # Header
    lines.append(line_fmt.format("", *headers))

    for label, metrics in report_dict.items():
        if isinstance(metrics, dict):
            row = [
                f"{metrics['precision'] * 100:>8.2f}",
                f"{metrics['recall'] * 100:>8.2f}",
                f"{metrics['f1-score'] * 100:>8.2f}",
                f"{int(metrics['support']):>8}"
            ]
            lines.append(line_fmt.format(str(label), *row))

    # Accuracy row
    if 'accuracy' in report_dict:
        accuracy = report_dict['accuracy'] * 100
        lines.append("\naccuracy  {:.2f}%".format(accuracy))

    return "\n".join(lines)

In [ ]:
y_test_cropped = y_test[:1000]  # For debugging, use a smaller subset
y_pred_cropped = model.predict(X_test[:1000])  # For debugging, use a smaller subset

report_str = classification_report_percent(y_test_cropped, y_pred_cropped, target_names=['negative', 'positive'])
print(report_str)

# report_full = classification_report_percent(y_test, model.predict(X_test), target_names=['negative', 'positive'])
# print(report_full)

In [ ]:
# Make predictions
y_pred_cropped = model.predict(X_test[:1000])  # For debugging, use a smaller subset

#y_pred = model.predict(X_test)

6. Model Saving

In [ ]:
# Save the trained model
model_path = f"{DATA_ROOT}/project2_data/{MODEL_TYPE}_model.pkl"
print(f"\nSaving model to {model_path}")
model.save_classifier(file_path=model_path)

# Save the vectorizer
vectorizer_path = f"{DATA_ROOT}/project2_data/tfidf_vectorizer.pkl"
with open(vectorizer_path, 'wb') as f:
    import pickle
    pickle.dump(vectorizer, f)
print(f"Saved vectorizer to {vectorizer_path}")
